# Lab 2: Build a tank inventory for each site

**Course:** Politics of World Energy Rivalry (Stanford, Policy)

**What you will do**

1. Download storage tank outlines from OpenStreetMap (OSM) for each AOI.
2. Compute each tank's diameter and rough capacity.
3. Check which tanks are big enough to see in 3 m PlanetScope imagery and in 0.5 m SkySat imagery.
4. Save one tank file per site for Lab 3.

**Why this matters for policy:** You cannot read fill levels until you know where the tanks are. OSM is a free, volunteer-built map. Its coverage differs by country. That gap is itself a finding: the sites with the least public data are often the ones where outside checks matter most.

## Concepts you need first

**OpenStreetMap (OSM).** A shared world map edited by volunteers. Storage tanks are drawn as closed lines (ways) and tagged `man_made=storage_tank`.

**Tag.** A label on a map feature, written as `key=value`. For example `content=oil` says the tank holds oil. Tags are optional, so many tanks have none.

**Overpass API.** A search service for OSM data. You send a query with a bounding box and a tag, and it returns matching features.

**Footprint.** The outline of a tank seen from above. A cylindrical tank has a circular footprint.

**Projected coordinates.** Latitude and longitude are angles, not meters. To measure area or distance we convert to a projection that uses meters. We use UTM, which has a zone for each part of the globe.

**Equivalent diameter.** If a footprint has area A, a circle with the same area has diameter 2 × √(A ÷ π). This works even when the outline is drawn roughly.

**Capacity.** A cylinder's volume is π × r² × height. Footprints give r. Height is not in OSM for most tanks, so we assume a value and state that assumption. One cubic meter holds about 6.29 barrels of oil.

**Pixels across a tank.** diameter ÷ pixel size. A tank needs several pixels across before software can find its shadow. We use 8 as a working minimum.

In [ ]:
import json
import math
import time
from pathlib import Path

import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape, Polygon

## Step 1: Settings

In [ ]:
REPO = Path("..").resolve()
AOI_DIR = REPO / "AOI"
TANK_DIR = REPO / "data" / "tanks"
TANK_DIR.mkdir(parents=True, exist_ok=True)

# Overpass servers. If one is busy the code tries the next.
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]
HEADERS = {"User-Agent": "power-radar-lab-class/1.0 (Stanford course lab)"}

MIN_DIAMETER_M = 15        # ignore anything smaller (small process tanks)
ASSUMED_HEIGHT_M = 15      # ASSUMPTION: typical large tank shell height
BARRELS_PER_M3 = 6.2898    # conversion factor
PIXEL_PLANETSCOPE_M = 3.0
PIXEL_SKYSAT_M = 0.5
MIN_PIXELS_ACROSS = 8      # ASSUMPTION: smallest tank we trust to measure

## Step 2: Load the AOIs

In [ ]:
aois = {}
for path in sorted(AOI_DIR.glob("*_aoi.geojson")):
    feature = json.loads(path.read_text())["features"][0]
    slug = path.stem.replace("_aoi", "")
    aois[slug] = {
        "name": feature["properties"]["facility"],
        "shape": shape(feature["geometry"]),
    }
print(len(aois), "sites loaded")

## Step 3: Ask Overpass for the tanks

The query says: inside this box, find every way tagged `man_made=storage_tank`, and give me its tags and its corner coordinates (`out tags geom`). The bounding box is the smallest rectangle around the AOI, given as south, west, north, east.

The function saves each answer to a file. If the file exists, it reads that instead of asking the server again. This is called **caching**. It keeps you from hitting the server over and over.

In [ ]:
def fetch_tanks_raw(slug, aoi_shape):
    """Return the raw Overpass answer for one AOI, using a cache file."""
    cache_file = TANK_DIR / f"raw_{slug}.json"
    if cache_file.exists():                              # use saved answer if present
        return json.loads(cache_file.read_text())

    west, south, east, north = aoi_shape.bounds          # shapely gives (minx, miny, maxx, maxy)
    query = f"""
    [out:json][timeout:90];
    way["man_made"="storage_tank"]({south},{west},{north},{east});
    out tags geom;
    """
    last_error = None
    for url in OVERPASS_URLS:                            # try each server in turn
        try:
            resp = requests.post(url, data={"data": query}, headers=HEADERS, timeout=120)
            resp.raise_for_status()
            result = resp.json()
            cache_file.write_text(json.dumps(result))    # save for next time
            return result
        except Exception as err:                         # network error, busy server, etc.
            last_error = err
            time.sleep(2)
    raise RuntimeError(f"All Overpass servers failed: {last_error}")


raw = {}
for slug, aoi in aois.items():
    raw[slug] = fetch_tanks_raw(slug, aoi["shape"])
    print(f"{aoi['name']:<35} {len(raw[slug]['elements']):>5} tank outlines")

## Step 4: Turn outlines into measurements

For each OSM way we:

1. build a polygon from its corner points,
2. keep only those inside the AOI,
3. project to meters,
4. compute area, equivalent diameter, and the center point,
5. estimate capacity with the assumed height.

`GeoDataFrame` is a table where one column holds shapes. It works like a spreadsheet with a map attached.

In [ ]:
def elements_to_gdf(slug, elements, aoi_shape):
    """Convert Overpass ways into a table of tank measurements."""
    records = []
    for el in elements:
        pts = [(p["lon"], p["lat"]) for p in el.get("geometry", [])]   # (x, y) pairs
        if len(pts) < 4 or pts[0] != pts[-1]:             # need a closed outline
            continue
        poly = Polygon(pts)
        if not poly.is_valid or not poly.centroid.within(aoi_shape):
            continue
        tags = el.get("tags", {})
        records.append({
            "osm_id": el["id"],
            "content": tags.get("content") or tags.get("substance"),   # may be empty
            "operator": tags.get("operator"),
            "geometry": poly,
        })

    if not records:
        return gpd.GeoDataFrame(columns=["osm_id", "geometry"], geometry="geometry", crs="EPSG:4326")

    gdf = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")

    # Project to meters so area and diameter mean something.
    utm = gdf.estimate_utm_crs()
    metric = gdf.to_crs(utm)
    gdf["area_m2"] = metric.area
    gdf["diameter_m"] = 2 * (gdf["area_m2"] / math.pi) ** 0.5   # equivalent diameter
    center = metric.centroid.to_crs("EPSG:4326")                # center back in lat/lon
    gdf["center_lon"] = center.x
    gdf["center_lat"] = center.y

    # Capacity of a cylinder with the assumed height.
    gdf["assumed_height_m"] = ASSUMED_HEIGHT_M
    gdf["capacity_m3"] = gdf["area_m2"] * ASSUMED_HEIGHT_M
    gdf["capacity_bbl"] = gdf["capacity_m3"] * BARRELS_PER_M3

    # How many pixels wide is each tank for each sensor?
    gdf["px_planetscope"] = gdf["diameter_m"] / PIXEL_PLANETSCOPE_M
    gdf["px_skysat"] = gdf["diameter_m"] / PIXEL_SKYSAT_M
    gdf["ok_planetscope"] = gdf["px_planetscope"] >= MIN_PIXELS_ACROSS
    gdf["ok_skysat"] = gdf["px_skysat"] >= MIN_PIXELS_ACROSS

    return gdf[gdf["diameter_m"] >= MIN_DIAMETER_M].reset_index(drop=True)


tanks = {}
for slug, aoi in aois.items():
    tanks[slug] = elements_to_gdf(slug, raw[slug]["elements"], aoi["shape"])
    # Save for Lab 3.
    out = TANK_DIR / f"tanks_{slug}.geojson"
    if len(tanks[slug]):
        tanks[slug].to_file(out, driver="GeoJSON")
    elif out.exists():
        out.unlink()          # remove an old file so Lab 3 does not read stale data

## Step 5: Summary by site

Read the table before you read the map. Check the tank count, the median diameter, and the share visible to each sensor.

In [ ]:
rows = []
for slug, gdf in tanks.items():
    n = len(gdf)
    rows.append({
        "site": aois[slug]["name"],
        "tanks": n,
        "median_diameter_m": round(gdf["diameter_m"].median(), 1) if n else None,
        "assumed_capacity_million_bbl": round(gdf["capacity_bbl"].sum() / 1e6, 1) if n else 0,
        "share_ok_planetscope": round(gdf["ok_planetscope"].mean(), 2) if n else None,
        "share_ok_skysat": round(gdf["ok_skysat"].mean(), 2) if n else None,
        "tagged_with_content": int(gdf["content"].notna().sum()) if n else 0,
    })
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, (slug, gdf) in zip(axes.ravel(), tanks.items()):
    # Draw the AOI outline, then the tanks on top.
    x, y = aois[slug]["shape"].exterior.xy
    ax.plot(x, y, color="black", linewidth=1)
    if len(gdf):
        gdf.plot(ax=ax, color="tab:blue", edgecolor="none")
    ax.set_title(f"{aois[slug]['name']}  ({len(gdf)} tanks)", fontsize=10)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of tank sizes. Vertical lines show the smallest diameter each sensor can use.
all_d = pd.concat([g["diameter_m"] for g in tanks.values() if len(g)])
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_d, bins=30, color="tab:blue")
ax.axvline(MIN_PIXELS_ACROSS * PIXEL_PLANETSCOPE_M, color="red", linestyle="--",
           label=f"PlanetScope minimum ({MIN_PIXELS_ACROSS * PIXEL_PLANETSCOPE_M:.0f} m)")
ax.axvline(MIN_PIXELS_ACROSS * PIXEL_SKYSAT_M, color="green", linestyle="--",
           label=f"SkySat minimum ({MIN_PIXELS_ACROSS * PIXEL_SKYSAT_M:.0f} m)")
ax.set_xlabel("Tank diameter (m)")
ax.set_ylabel("Number of tanks")
ax.legend()
plt.show()

## Your turn

1. Which site has the fewest OSM tanks? Does that match the "observability" rating in `data/Book1.csv`? What might explain a gap between the real number of tanks and the mapped number?
2. Change `ASSUMED_HEIGHT_M` to 10 and then 20. How much does the site's total capacity move? What does that tell you about how much to trust capacity estimates before you measure height?
3. Change `MIN_PIXELS_ACROSS` to 5 and then 12. How does the share of usable tanks change for PlanetScope?
4. OSM `content` tags are sparse. If a tank has no `content` tag, can you say it holds crude oil? What other evidence could you add?

**If a site has zero tanks:** Lab 3 includes a tank finder that works directly on imagery. You will use it for those sites.

**Next:** `03_tank_metrics_from_imagery.ipynb` measures fill level and volume.